## Import dependencies

In [57]:
!pip install bs4 requests dotenv
!pip install lxml --break-system-packages
!pip install pandas
from bs4 import BeautifulSoup
import requests
import os
from dotenv import load_dotenv
import pandas as pd
load_dotenv()

print("Dependencies Imported")

Dependencies Imported


## Log in to CSES

In [58]:
LOGIN_URL = "https://cses.fi/login"
TARGET_URL = "https://cses.fi/problemset/"
session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
})

In [59]:
login_page = session.get("https://cses.fi/login")
soup = BeautifulSoup(login_page.text, "lxml")

csrf_token = soup.find("input", {"name": "csrf_token"})["value"]

login_data = {
    "csrf_token": csrf_token,
    "nick": os.getenv("CSES_USERNAME"),
    "pass": os.getenv("CSES_PASSWORD"),
}

In [60]:
login_response = session.post(LOGIN_URL, data=login_data)

In [61]:
print(login_response.url)
check = session.get("https://cses.fi/")
soup_check = BeautifulSoup(check.text, "lxml")
print(soup_check.find("a", href=lambda h: h and "logout" in h))

https://cses.fi/
<a href="/logout" title="Log out"><i aria-label="Log out" class="fas fa-sign-out-alt"></i><span>Log out</span></a>


In [62]:
target_response = session.get(TARGET_URL)

In [63]:
target_response.text

'<!DOCTYPE html>\n<html>\n<head>\n  <meta charset="UTF-8">\n  <meta name="viewport" content="width=device-width,initial-scale=1">\n  <link rel="stylesheet " type="text/css" href="/cses.css?1" id="styles">\n  <link rel="stylesheet alternate" type="text/css" href="/cses-dark.css?1" id="styles-dark">\n  <meta name="theme-color" content="white" id="theme-color">\n  <script type="application/json" id="darkmode-enabled">false</script>\n  <script src="/ui.js"></script>\n  <link rel="stylesheet" type="text/css" href="/lib/fontawesome/css/all.min.css">\n  <title>CSES - CSES Problem Set - Tasks</title></head>\n<body class="with-sidebar norm-width">\n  <div class="header">\n    <div>\n      <a href="/" class="logo"><img src="/logo.png?1" alt="CSES"></a>\n      <a class="menu-toggle" onclick="document.body.classList.toggle(\'menu-open\');">\n        <i class="fas fa-bars"></i>\n      </a>\n      <div class="controls">\n                <a class="account" href="/user/321052">Esmail</a>\n        <spa

In [64]:
soup = BeautifulSoup(target_response.text, 'lxml')

In [65]:
headers = soup.select('h2')
headers.pop(0)
print(headers)

[<h2>Introductory Problems</h2>, <h2>Sorting and Searching</h2>, <h2>Dynamic Programming</h2>, <h2>Graph Algorithms</h2>, <h2>Range Queries</h2>, <h2>Tree Algorithms</h2>, <h2>Mathematics</h2>, <h2>String Algorithms</h2>, <h2>Geometry</h2>, <h2>Advanced Techniques</h2>, <h2>Sliding Window Problems</h2>, <h2>Interactive Problems</h2>, <h2>Bitwise Operations</h2>, <h2>Construction Problems</h2>, <h2>Advanced Graph Problems</h2>, <h2>Counting Problems</h2>, <h2>Additional Problems I</h2>, <h2>Additional Problems II</h2>]


## Parse HTML into a dictionary + dataframe

In [66]:
# Segment name: [(Problem name, Solved by me, total solved, total attempts)]
ans = {}

df = pd.DataFrame({
    'Header': [],
    'Problem Name': [],
    'Solved By Me': [],
    'Solve Count' : [],
    'Attempt Count': [] 
})


for header in headers:
    n = header.text

    ans[n] = []
    task_list = header.next_sibling.select('li')
    tasks = task_list
    for task in tasks:
        done = True if len(task.select('span.full')) >= 1 else False
        task_name = task.select_one('a').text
        
        solve_count_numbers = task.select_one('span.detail').text

        solve_count_numbers = solve_count_numbers.split('/')

        attempts = solve_count_numbers[-1].strip()
        solve_count = solve_count_numbers[0].strip()

        solve_count, attempts = int(solve_count), int(attempts)

        row = (task_name, done, solve_count, attempts)

        ans[n].append(row)

        row = pd.DataFrame({
            'Header': [n],
            'Problem Name': [row[0]],
            'Solved By Me': [row[1]],
            'Solve Count' : [row[2]],
            'Attempt Count': [row[3]]
        })

        df = pd.concat([df, row], ignore_index=True)

df['Header'] = df['Header'].astype(str)
df['Problem Name'] = df['Problem Name'].astype(str)
df['Solved By Me'] = df['Solved By Me'].astype(bool)
df['Solve Count'] = df['Solve Count'].astype(int)
df['Attempt Count'] = df['Attempt Count'].astype(int)

ans

{'Introductory Problems': [('Weird Algorithm', True, 170410, 178106),
  ('Missing Number', True, 147213, 153960),
  ('Repetitions', True, 127983, 132868),
  ('Increasing Array', True, 120604, 124678),
  ('Permutations', True, 105797, 108879),
  ('Number Spiral', True, 75000, 81419),
  ('Two Knights', True, 57790, 59514),
  ('Two Sets', True, 62810, 67278),
  ('Bit Strings', True, 71575, 75392),
  ('Trailing Zeros', True, 66372, 70452),
  ('Coin Piles', True, 59003, 64262),
  ('Palindrome Reorder', True, 54817, 57727),
  ('Gray Code', True, 37332, 41672),
  ('Tower of Hanoi', True, 34762, 36093),
  ('Creating Strings', True, 46238, 47373),
  ('Apple Division', True, 46826, 52593),
  ('Chessboard and Queens', True, 28497, 28978),
  ('Raab Game I', True, 6892, 7773),
  ('Mex Grid Construction', True, 6712, 7044),
  ('Knight Moves Grid', True, 6771, 6921),
  ('Grid Coloring I', True, 5691, 5892),
  ('Digit Queries', True, 20047, 23046),
  ('String Reorder', True, 5120, 5871),
  ('Grid Path

In [67]:
df

,Header,Problem Name,Solved By Me,Solve Count,Attempt Count
0,Introductory Problems,Weird Algorithm,True,170410,178106
1,Introductory Problems,Missing Number,True,147213,153960
2,Introductory Problems,Repetitions,True,127983,132868
3,Introductory Problems,Increasing Array,True,120604,124678
4,Introductory Problems,Permutations,True,105797,108879
...,...,...,...,...,...
395,Additional Problems II,Maximum Building II,False,475,584
396,Additional Problems II,Stick Divisions,False,4035,4712
397,Additional Problems II,Stick Difference,False,82,296
398,Additional Problems II,Coding Company,False,1605,2096


## Analysis

In [75]:
data = df.copy()

In [76]:
print("You have solved: ", data['Solved By Me'].sum(), "/", len(data), ".")
print()
print("Solve Count By Category: \n")
display(data.groupby('Header')['Solved By Me'].sum().sort_values(ascending=False))

You have solved:  160 / 400 .

Solve Count By Category: 



Header
Sorting and Searching      35
Graph Algorithms           26
Introductory Problems      23
Dynamic Programming        22
Range Queries              15
Mathematics                12
Sliding Window Problems     9
Tree Algorithms             8
String Algorithms           3
Advanced Graph Problems     2
Advanced Techniques         2
Additional Problems I       1
Additional Problems II      1
Bitwise Operations          1
Geometry                    0
Counting Problems           0
Construction Problems       0
Interactive Problems        0
Name: Solved By Me, dtype: int64

In [78]:
data['Difficulty'] = data['Solve Count'] / data['Attempt Count']

data['Difficulty'] = data['Difficulty'].apply(lambda x: str(round(x*100 *100) / 100) + "%")

In [82]:
print("Should solve next: \n")

unsolved = data.loc[data['Solved By Me'] == False]

unsolved.sort_values(by=['Solve Count', 'Difficulty'], ascending=False).head(10)

Should solve next: 



,Header,Problem Name,Solved By Me,Solve Count,Attempt Count,Difficulty
146,Tree Algorithms,Tree Distances I,False,22181,23757,93.37%
147,Tree Algorithms,Tree Distances II,False,18088,18898,95.71%
169,Mathematics,Binomial Coefficients,False,11797,13035,90.5%
23,Introductory Problems,Grid Path Description,False,11209,14395,77.87%
151,Tree Algorithms,Counting Paths,False,10804,11411,94.68%
155,Tree Algorithms,Distinct Colors,False,9926,10872,91.3%
217,Geometry,Point Location Test,False,8294,9213,90.02%
102,Graph Algorithms,Planets Cycles,False,8083,8957,90.24%
156,Tree Algorithms,Finding a Centroid,False,8048,8372,96.13%
179,Mathematics,Fibonacci Numbers,False,7786,9698,80.28%


## Output to JSON

In [ ]:
df.to_json('data/output.json')